In [1]:
import atoti as tt
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

load_dotenv()

engine = create_engine(
    f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}"
)
print("Engine ready.")

Welcome to Atoti 0.9.14!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.
Engine ready.


In [2]:
fact = pd.read_sql("SELECT * FROM fact_post", engine)
dim_time = pd.read_sql("SELECT * FROM dim_time", engine)
dim_platform = pd.read_sql("SELECT * FROM dim_platform", engine)
dim_topic = pd.read_sql("SELECT * FROM dim_topic", engine)
dim_sentiment = pd.read_sql("SELECT * FROM dim_sentiment", engine)
print("Data loaded.")

print(
    fact.shape,
    dim_time.shape,
    dim_platform.shape,
    dim_topic.shape,
    dim_sentiment.shape
)

print("\nfact columns:", fact.columns.tolist())

print("FACT")
print(fact.head())

print("\nDIM TIME")
print(dim_time.head())

print("\nDIM PLATFORM")
print(dim_platform.head())

print("\nDIM TOPIC")
print(dim_topic.head())

print("\nDIM SENTIMENT")
print(dim_sentiment.head())

Data loaded.
(7311, 17) (2922, 11) (5, 3) (11, 4) (9, 3)

fact columns: ['post_id', 'source_id', 'source_url', 'time_id', 'platform_id', 'topic_id', 'sentiment_id', 'like_count', 'comment_count', 'quote_count', 'retweet_count', 'upvote_count', 'downvote_count', 'sentiment_score', 'engagement_tier', 'posted_at', 'loaded_at']
FACT
   post_id            source_id  \
0     7677  1741609786356179429   
1     7678  1741599474919821483   
2     7679  1741607826404036728   
3     7680  1741607016853127220   
4     7681  1741606336863903818   

                                          source_url  time_id  platform_id  \
0  https://x.com/@AzamFiqri1/status/1741609786356...      365            1   
1  https://x.com/@rodi_mufrodi/status/17415994749...      365            1   
2  https://x.com/@crazygirl2331/status/1741607826...      365            1   
3  https://x.com/@Miyar__/status/1741607016853127220      365            1   
4  https://x.com/@5ft10Man/status/174160633686390...      365       

In [3]:
df = fact.merge(dim_time, on="time_id") \
         .merge(dim_platform, on="platform_id") \
         .merge(dim_topic, on="topic_id") \
         .merge(dim_sentiment, on="sentiment_id")
print("Merge complete:", df.shape)

Merge complete: (7308, 34)


In [4]:
df["top_keywords"] = df["top_keywords"].apply(
    lambda x: ", ".join(x) if isinstance(x, list) else str(x) if x is not None else ""
)
df["year"] = df["year"].astype(str)
df["quarter"] = df["quarter"].apply(lambda x: f"Q{x}")
df["month_num"] = df["month"].astype(str).str.zfill(2)
df["day_of_month"] = df["day_of_month"].astype(str)

# Drop not-used-columns
df = df.drop(columns=["time_id","platform_id","topic_id","sentiment_id",
                       "loaded_at","source_url","posted_at"])
print("Fixes complete.")
print(df.dtypes)

Fixes complete.
post_id                int64
source_id             object
like_count             int64
comment_count          int64
quote_count            int64
retweet_count          int64
upvote_count           int64
downvote_count         int64
sentiment_score      float64
engagement_tier       object
full_date             object
year                  object
quarter               object
month                  int64
month_name            object
week_of_year           int64
day_of_month          object
day_of_week            int64
day_name              object
is_weekend              bool
platform_name         object
channel               object
topic_label           object
topic_category        object
top_keywords          object
sentiment_label       object
confidence_bucket     object
month_num             object
dtype: object


In [5]:
df["week_of_year"] = df["week_of_year"].astype(str).apply(lambda x: f"W{x}")
df["day_of_week"] = df["day_of_week"].astype(str)
df["month"] = df["month_name"]  # Using month_name, drop numbers
df = df.drop(columns=["month_num"])

print("Time dimensions ready.")

Time dimensions ready.


In [6]:
session = tt.Session.start(
    tt.SessionConfig(
        port=9090,
        user_content_storage="./atoti_save"
    )
)

print("Session ready.")

Session ready.


In [7]:
store = session.read_pandas(
    df,
    table_name="Posts",
    keys=["post_id"]
)
print("Store loaded.")

Store loaded.


In [8]:
cube = session.create_cube(store, "SocioEconomic")
print("Cube ready.")

Cube ready.


In [9]:
h, l, m = cube.hierarchies, cube.levels, cube.measures

h["Time"] = {
    "Year": l[("Posts", "year", "year")],
    "Quarter": l[("Posts", "quarter", "quarter")],
    "Month": l[("Posts", "month_name", "month_name")],
    "Week": l[("Posts", "week_of_year", "week_of_year")],
    "Day": l[("Posts", "day_of_month", "day_of_month")]
}

h["Platform"] = {
    "Platform": l[("Posts", "platform_name", "platform_name")],
    "Channel": l[("Posts", "channel", "channel")]
}

h["Topic"] = {
    "Category": l[("Posts", "topic_category", "topic_category")],
    "Topic": l[("Posts", "topic_label", "topic_label")]
}

h["Sentiment"] = {
    "Label": l[("Posts", "sentiment_label", "sentiment_label")],
    "Confidence": l[("Posts", "confidence_bucket", "confidence_bucket")]
}

h["Engagement Tier"] = {
    "Tier": l[("Posts", "engagement_tier", "engagement_tier")]
}

h["Day Type"] = {
    "Is Weekend": l[("Posts", "is_weekend", "is_weekend")],
    "Day Name": l[("Posts", "day_name", "day_name")]
}

print("Hierarchies ready.")

Hierarchies ready.


In [10]:
m["Avg Sentiment Score"] = tt.agg.mean(store["sentiment_score"])
m["Total Likes"] = tt.agg.sum(store["like_count"])
m["Total Comments"] = tt.agg.sum(store["comment_count"])
m["Total Retweets"] = tt.agg.sum(store["retweet_count"])
m["Total Quotes"] = tt.agg.sum(store["quote_count"])
m["Total Upvotes"] = tt.agg.sum(store["upvote_count"])
m["Total Downvotes"] = tt.agg.sum(store["downvote_count"])
m["Total Engagement"] = tt.agg.sum(
    store["like_count"] + store["comment_count"] +
    store["retweet_count"] + store["quote_count"] +
    store["upvote_count"]
)
print("Measures ready.")

Measures ready.


In [11]:
print(session.link)

http://localhost:9090


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3EMyaoa6bwbK17CbY20oMMhxahA_qkuxJZcFAjqcyCUKdhaB")
ngrok.kill()

public_url = ngrok.connect(9090, bind_tls=True)
url_string = public_url.public_url
print("Public link:", url_string)

Public link: https://expenses-headphone-enforced.ngrok-free.dev


t=2026-06-04T09:47:54+0700 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=8f149d4c4a34 clientid=62ded517fb6972c39b9bab1d71c8dbdc
t=2026-06-04T09:47:54+0700 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=f268b35ff6cd err="session closed"
t=2026-06-04T09:48:06+0700 lvl=eror msg="session closed, starting reconnect loop" obj=tunnels.session obj=csess id=f268b35ff6cd err="read tcp 192.168.0.108:50336->3.20.27.198:443: wsarecv: An established connection was aborted by the software in your host machine."
t=2026-06-04T09:48:12+0700 lvl=warn msg="failed to check for update" obj=updater err="Post \"https://update.ngrok-agent.com/check\": context deadline exceeded"
t=2026-06-04T09:48:24+0700 lvl=eror msg="heartbeat timeout, terminating session" obj=tunnels.session obj=csess id=7f8b2a0077e9 clientid=62ded517fb6972c39b9bab1d71c8dbdc
